# Clase 8 — Autocorrelación espacial: ¿el patrón es azar?

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 8 — Geoestadística exploratoria |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿El NBI se distribuye al azar en la ciudad, o los radios se parecen a sus vecinos?

En la Clase 7 construimos una tabla con el porcentaje de hogares con necesidades básicas
insatisfechas de cada uno de los 3.554 radios censales de la Ciudad de Buenos Aires. Sabemos
cuánto vale la variable en cada unidad, y en la Clase 5 aprendimos a mapearla.

Falta la pregunta que un mapa no contesta: **¿lo que se ve es un patrón o es ruido?**

Cualquier mapa de cualquier variable parece tener estructura. El ojo agrupa manchas de color
aunque los valores se hayan repartido al azar. Para afirmar que hay un patrón hace falta
compararlo contra lo que se vería si no lo hubiera, y eso es lo que hace la clase de hoy.

Es además el primer encuentro del seminario donde **la hipótesis nula es espacial**: "esta
variable se distribuyó al azar en el territorio". Todo lo anterior —construir variables,
medirlas, mapearlas— fue preparación para poder formularla.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Explicar** qué es la autocorrelación espacial y por qué invalida el supuesto de
   independencia de las observaciones.
2. **Construir** una matriz de pesos espaciales y justificar el criterio de vecindad elegido.
3. **Calcular e interpretar** el I de Moran global y su prueba por permutaciones.
4. **Identificar** conglomerados locales con LISA y distinguir los cuatro tipos de asociación.
5. **Medir** la autocorrelación de una variable categórica con Join-Count.
6. **Leer** un semivariograma: nugget, meseta y rango.
7. **Distinguir** una asociación espacial de una explicación causal.

## 3. Material de esta clase

| Archivo | Contenido | Fuente |
|---|---|---|
| `caba_radios_2022.gpkg` | 3.554 radios censales con población, hogares y hogares con NBI | BA Data — Dirección General de Estadísticas y Censos, Censo 2022 |

Es la misma capa de la Clase 7. Todo el trabajo de hoy se hace sobre ella.

**Bibliotecas nuevas:** `libpysal` construye las matrices de pesos espaciales y `esda` calcula
los estadísticos. Las dos pertenecen a **PySAL**, la biblioteca de análisis espacial de Python.

---

## 4. Qué es la autocorrelación espacial

**Conceptos clave.** La **autocorrelación** es la correlación de una variable consigo misma. En
una serie de tiempo, la temperatura de hoy se parece a la de ayer: la variable está
correlacionada con su propio pasado. En el espacio ocurre lo mismo, pero en todas las
direcciones a la vez: el valor de una unidad se parece al de las unidades que la rodean.

Se llama **autocorrelación espacial positiva** cuando los valores altos están cerca de otros
altos y los bajos cerca de otros bajos —el patrón que solemos llamar conglomerado o *cluster*—.
Se llama **negativa** cuando ocurre lo contrario, valores altos rodeados de bajos, que produce
un patrón alternado, como un tablero de ajedrez. Y no hay autocorrelación cuando saber el valor
de una unidad no informa nada sobre el de sus vecinas.

### 4.1 La primera ley de la geografía

En la Clase 2 enunciamos la formulación de Waldo Tobler:

> *"Todo está relacionado con todo lo demás, pero las cosas cercanas están más relacionadas
> entre sí que las cosas lejanas."*

Es una regularidad empírica, no un teorema, y es la razón por la que el análisis espacial tiene
sentido. Si las cosas cercanas no se parecieran más que las lejanas, la posición no aportaría
ninguna información y no habría nada que analizar.

Medir la autocorrelación es, en el fondo, **poner un número a la primera ley** para una variable
y un territorio concretos.

### 4.2 Por qué importa: dos motivos distintos

Hay dos razones para medir autocorrelación espacial, y conviene no confundirlas porque llevan a
conclusiones opuestas.

**Como estorbo.** Casi toda la estadística que conocen supone que las observaciones son
independientes. Si los radios vecinos se parecen, esa independencia no existe: la muestra tiene
menos información de la que su tamaño sugiere. Los errores estándar salen demasiado chicos y las
pruebas de hipótesis declaran significativo lo que no lo es. Acá el objetivo es **corregir**.

**Como sustancia.** A veces el mecanismo que se estudia *es* espacial: la difusión de una
práctica entre barrios vecinos, la segregación residencial, el efecto de vecindario, la
accesibilidad a un servicio. Acá la autocorrelación no es un problema a corregir sino **el
objeto mismo de la investigación**.

La clase de hoy mide. Qué se hace después con esa medida depende de cuál de las dos preguntas se
esté haciendo.

---

## 5. Preparación

### ▶️ Bibliotecas

In [ ]:
!pip install -q "geopandas==1.0.1" "mapclassify==2.8.1" "matplotlib==3.9.2" \
               "libpysal==4.12.1" "esda==2.6.0"

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import libpysal
import esda

print("Bibliotecas listas")

### ▶️ Los radios censales

La misma capa de la Clase 7. Se reproyecta una vez a EPSG:5347, porque la matriz de pesos por
distancia y el semivariograma miden en metros.

In [ ]:
DATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/"
METRICO = "EPSG:5347"

radios_m = gpd.read_file(DATOS + "caba_radios_2022.gpkg").to_crs(METRICO)

print(f"{len(radios_m)} radios censales")
radios_m.head(3)

### ▶️ La variable

El porcentaje de hogares con al menos una necesidad básica insatisfecha. Dos radios no tienen
hogares —son parques o zonas de uso no residencial— y producirían una división por cero, así que
quedan fuera del análisis.

In [ ]:
radios_m = radios_m[radios_m["hogares"] > 0].copy()
radios_m["perc_nbi"] = radios_m["hogares_nbi"] / radios_m["hogares"] * 100

print(f"Radios con hogares: {len(radios_m)}")
radios_m["perc_nbi"].describe().round(1)

### 👀 La variable en el mapa

Antes de medir nada, mirar. La distribución es muy asimétrica —la mediana es 1,9 % y el máximo
supera el 88 %—, de modo que el criterio de clasificación importa: con intervalos iguales casi
toda la ciudad quedaría en la clase más clara. Usamos quiebres naturales, como en la Clase 5.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(15, 7))

radios_m.plot(column="perc_nbi", cmap="Reds", scheme="FisherJenks", k=5, legend=True,
              edgecolor="white", linewidth=0.05, ax=ejes[0],
              legend_kwds=dict(title="% hogares con NBI", loc="upper left", fontsize=8))
ejes[0].set_title("NBI por radio censal, CABA 2022")
ejes[0].set_axis_off()

ejes[1].hist(radios_m["perc_nbi"], bins=60, color="#c44e52")
ejes[1].set_xlabel("% de hogares con NBI")
ejes[1].set_ylabel("Radios")
ejes[1].set_title("Distribución de la variable")

plt.tight_layout()
plt.show()

### 🔍 Lo que sugiere el mapa

Se ve una diferencia marcada entre el sur y el norte de la ciudad, y algunas manchas oscuras muy
localizadas que corresponden a barrios populares.

Pero *verlo* no alcanza. El ojo humano encuentra estructura en cualquier mapa, incluso en uno
construido con valores repartidos al azar, y además el patrón que uno cree ver depende del
criterio de clasificación que eligió. Todo lo que sigue existe para transformar esa impresión en
una afirmación que se pueda sostener.

---

## 6. La matriz de pesos espaciales

**Conceptos clave.** Para decir que una unidad "se parece a sus vecinas" hay que definir primero
**quién es vecino de quién**. Esa definición se expresa en una **matriz de pesos espaciales**
$W$: una tabla de $n \times n$ donde el elemento $w_{ij}$ indica cuánto pesa la unidad $j$ como
vecina de la unidad $i$. Por convención $w_{ii} = 0$: ninguna unidad es vecina de sí misma.

La matriz no viene con los datos. **Es una decisión del analista**, y es la decisión más
importante de toda la clase: cambiarla cambia el resultado.

Los criterios más usados son tres:

| Criterio | Define como vecinas a… | Cuándo conviene |
|---|---|---|
| **Contigüidad** | Las unidades que comparten frontera | Unidades areales que teselan el territorio: radios, departamentos, provincias |
| **K vecinos más próximos** (KNN) | Las *k* unidades más cercanas, siempre *k* | Puntos, o áreas de tamaño muy desigual |
| **Banda de distancia** | Todas las que están a menos de *d* | Cuando existe una distancia con sentido sustantivo |

Dentro de la contigüidad hay dos variantes, que se nombran por el movimiento de las piezas de
ajedrez: **Rook** (torre) exige compartir un segmento de frontera, y **Queen** (reina) admite
además el contacto en un solo vértice. Queen es la opción habitual para radios censales, porque
en una trama urbana amanzanada hay muchos encuentros en esquina.

### ▶️ Contigüidad Queen

`libpysal` la construye a partir de las geometrías. El objeto resultante guarda, para cada
radio, la lista de sus vecinos.

In [ ]:
w_queen = libpysal.weights.Queen.from_dataframe(radios_m, use_index=False)

vecinos = [len(v) for v in w_queen.neighbors.values()]

print(f"Radios: {w_queen.n}")
print(f"Vecinos por radio — mínimo: {min(vecinos)} · promedio: {np.mean(vecinos):.1f} · máximo: {max(vecinos)}")
print(f"Radios sin ningún vecino (islas): {len(w_queen.islands)}")

### ✅ Comprobación — las islas

Un radio sin vecinos es un problema: los estadísticos que siguen no pueden calcular un promedio
de vecinos que no existe, y PySAL emite una advertencia y devuelve valores indefinidos.

Acá no hay ninguno, porque los radios censales teselan la ciudad sin huecos. Cuando aparecen
—una isla real, un enclave, un error de digitalización en el borde—, hay que decidir qué hacer y
**declararlo**: excluirlos del análisis, o cambiar a un criterio KNN, que por construcción nunca
deja a nadie sin vecinos.

### 👀 Qué significa "vecino"

La manera más rápida de entender una matriz de pesos es mirarla. Tomamos un radio cualquiera y
pintamos los que la matriz considera sus vecinos.

In [ ]:
i = 1500                                   # un radio cualquiera
vecinos_de_i = w_queen.neighbors[i]

fig, eje = plt.subplots(figsize=(8, 8))
zona = radios_m.iloc[[i]].buffer(1200).total_bounds

radios_m.plot(ax=eje, facecolor="white", edgecolor="grey", linewidth=0.4)
radios_m.iloc[vecinos_de_i].plot(ax=eje, facecolor="#9ecae1", edgecolor="#2b6ca3")
radios_m.iloc[[i]].plot(ax=eje, facecolor="#c44e52", edgecolor="black", linewidth=1.2)

eje.set_xlim(zona[0], zona[2]); eje.set_ylim(zona[1], zona[3])
eje.set_title(f"El radio {i} (rojo) y sus {len(vecinos_de_i)} vecinos Queen (azul)")
eje.set_axis_off()
plt.show()

### ▶️ Otro criterio, otra vecindad

Con KNN el mismo radio tiene otros vecinos: exactamente ocho, elegidos por cercanía entre
centroides y sin mirar si comparten frontera.

In [ ]:
w_knn = libpysal.weights.KNN.from_dataframe(radios_m, k=8)

print("Vecinos del radio", i)
print("  Queen:", sorted(w_queen.neighbors[i]))
print("  KNN 8:", sorted(w_knn.neighbors[i]))
print("  En ambos:", sorted(set(w_queen.neighbors[i]) & set(w_knn.neighbors[i])))

### ▶️ Estandarización por filas

Falta un paso. Con la matriz en crudo, un radio con 24 vecinos suma 24 veces y uno con 2 suma
dos: el resultado quedaría dominado por las unidades mejor conectadas.

La **estandarización por filas** (`transform = "R"`) divide cada peso por la cantidad de vecinos
de esa unidad, de modo que cada fila suma 1. Así $\sum_j w_{ij} x_j$ pasa a ser directamente
**el promedio de los vecinos**, que es lo que queremos comparar contra el valor propio.

In [ ]:
w_queen.transform = "R"

print("Pesos del radio", i, "después de estandarizar:")
print(" ", {v: round(p, 3) for v, p in zip(w_queen.neighbors[i], w_queen.weights[i])})
print("Suman:", round(sum(w_queen.weights[i]), 3))

---

## 7. El I de Moran global

**Conceptos clave.** El **I de Moran** resume en un solo número cuánta autocorrelación espacial
hay en toda la zona de estudio. Su fórmula es:

$$I = \frac{n}{\sum_i \sum_j w_{ij}} \cdot
      \frac{\sum_i \sum_j w_{ij}\,(x_i - \bar{x})(x_j - \bar{x})}
            {\sum_i (x_i - \bar{x})^2}$$

Conviene leerla de derecha a izquierda. El numerador de la segunda fracción multiplica el
desvío de cada unidad respecto de la media, $(x_i - \bar{x})$, por el desvío de cada una de sus
vecinas, $(x_j - \bar{x})$. Si una unidad está por encima de la media y sus vecinas también, el
producto es positivo. Si está por encima y sus vecinas por debajo, es negativo. El denominador
es la varianza, que normaliza la escala. Y la primera fracción corrige por el tamaño de la
matriz de pesos.

Es, estructuralmente, **un coeficiente de correlación entre la variable y el promedio de sus
vecinos**, y por eso se interpreta parecido: valores cercanos a +1 indican conglomerados fuertes,
cercanos a −1 un patrón alternado, y cercanos a 0 ausencia de estructura.

**Acá se ve por qué la matriz de pesos importa tanto:** $w_{ij}$ aparece dos veces en la
fórmula. Sin definirla, el índice no existe.

### 🧭 Qué significa "cero"

El valor de referencia no es exactamente 0. Bajo la hipótesis nula de distribución aleatoria, el
valor esperado del índice es

$$E[I] = \frac{-1}{n - 1}$$

un número negativo muy chico, que con miles de unidades es prácticamente cero pero no lo es. La
razón es que cada unidad se compara con las demás excluyéndose a sí misma, y eso introduce un
sesgo negativo leve.

Lo que hay que preguntarse no es si $I$ es distinto de cero, sino **si es lo bastante distinto
de $E[I]$ como para no atribuirlo al azar**.

### ▶️ El cálculo

In [ ]:
y = radios_m["perc_nbi"].values

moran = esda.Moran(y, w_queen, permutations=999)

print(f"I de Moran observado : {moran.I:.3f}")
print(f"Valor esperado E[I]  : {moran.EI:.5f}")
print(f"Pseudo p-valor       : {moran.p_sim:.3f}")
print(f"z-score              : {moran.z_sim:.1f}")

### 🧭 De dónde sale el p-valor: la prueba por permutaciones

El p-valor no viene de una distribución teórica sino de una **simulación**, y el procedimiento es
transparente:

1. Se toman los valores observados de la variable y **se reparten al azar** entre los radios,
   manteniendo intacta la geometría y la matriz de vecinos.
2. Se calcula el I de Moran de esa asignación aleatoria.
3. Se repite 999 veces.

Eso produce una distribución de referencia: **cómo sería el I de Moran si el NBI no tuviera nada
que ver con la posición**. El pseudo p-valor es la proporción de esas 999 simulaciones que
alcanzaron un valor tan extremo como el observado.

Es exactamente la lógica de una prueba de hipótesis, con la ventaja de que la hipótesis nula se
construye acá, a la vista, en lugar de suponer una distribución.

### 👀 El valor observado contra las 999 simulaciones

In [ ]:
fig, eje = plt.subplots(figsize=(9, 5))

eje.hist(moran.sim, bins=40, color="#bdbdbd", edgecolor="white")
eje.axvline(moran.EI, color="black", linestyle=":", linewidth=1.5, label=f"E[I] = {moran.EI:.4f}")
eje.axvline(moran.I, color="#c44e52", linewidth=2.5, label=f"I observado = {moran.I:.3f}")

eje.set_xlabel("I de Moran")
eje.set_ylabel("Simulaciones")
eje.set_title("El valor observado frente a 999 asignaciones al azar")
eje.legend()
plt.tight_layout()
plt.show()

El valor observado ni siquiera entra en el gráfico junto a la nube de simulaciones: ninguna de
las 999 reparticiones al azar se acercó. Eso es lo que significa un pseudo p-valor de 0,001, que
es el mínimo alcanzable con 999 permutaciones.

### 🧭 El diagrama de Moran

Hay una segunda manera de leer el índice, y es la que más ayuda a entenderlo.

Para cada radio se calcula el **rezago espacial** de la variable: el promedio de sus vecinos,
que es justamente $\sum_j w_{ij} x_j$ con la matriz estandarizada. Después se grafica el valor
propio contra ese promedio.

Si hay autocorrelación positiva, los puntos se alinean en una recta ascendente, y **la pendiente
de esa recta es el I de Moran**. El índice no es una abstracción: es la pendiente de una regresión
que se puede dibujar.

In [ ]:
from libpysal.weights import lag_spatial

y_est = (y - y.mean()) / y.std()          # variable estandarizada
lag_est = lag_spatial(w_queen, y_est)     # promedio de los vecinos

fig, eje = plt.subplots(figsize=(7, 7))
eje.scatter(y_est, lag_est, s=6, color="#4c72b0", alpha=0.4)
eje.axvline(0, color="grey", linewidth=0.8)
eje.axhline(0, color="grey", linewidth=0.8)

pendiente = np.polyfit(y_est, lag_est, 1)[0]
eje.plot(y_est, pendiente * y_est, color="#c44e52", linewidth=2)

eje.set_xlabel("NBI del radio (estandarizado)")
eje.set_ylabel("NBI promedio de sus vecinos")
eje.set_title(f"Diagrama de Moran — pendiente = {pendiente:.3f}")
plt.tight_layout()
plt.show()

### 🔍 Interpretación

**I = 0,543, con un pseudo p-valor de 0,001.** El NBI de la Ciudad de Buenos Aires no está
repartido al azar: los radios se parecen a sus vecinos mucho más de lo que se parecerían si la
variable se hubiera distribuido por sorteo.

La pendiente del diagrama coincide con el índice, como corresponde. Y la nube muestra algo que
el número solo no dice: la mayor parte de los radios se amontona cerca del origen —valores bajos
rodeados de valores bajos— y hay una cola larga hacia el cuadrante superior derecho, donde los
valores altos conviven con vecinos altos.

Dos advertencias sobre este resultado.

La primera es que **0,543 es un promedio de toda la ciudad**, y un promedio puede ocultar
comportamientos opuestos en distintas zonas. La sección siguiente desagrega eso.

La segunda es que el índice **depende de la matriz de pesos**. Con KNN en lugar de contigüidad,
o con otro valor de *k*, el número cambia. No es un defecto del método: es la consecuencia de que
"vecino" sea una definición y no un dato. Lo que corresponde es declarar el criterio usado, y
verificar que la conclusión no se invierta al cambiarlo.

In [ ]:
# El mismo cálculo con el otro criterio de vecindad
w_knn.transform = "R"
moran_knn = esda.Moran(y, w_knn, permutations=999)

print(f"Queen  : I = {moran.I:.3f}  (p = {moran.p_sim:.3f})")
print(f"KNN k=8: I = {moran_knn.I:.3f}  (p = {moran_knn.p_sim:.3f})")

---

## 8. LISA: dónde están los conglomerados

**Conceptos clave.** El I de Moran global responde *si* hay estructura, pero no *dónde*. Un solo
número para toda la ciudad puede resultar de un patrón parejo o de dos o tres zonas muy marcadas
sobre un fondo sin estructura.

Los **LISA** (*Local Indicators of Spatial Association*) descomponen el índice global en una
contribución por unidad. Para cada radio se calcula un $I_i$ local que compara su valor con el
promedio de sus vecinos, y se lo somete a la misma prueba por permutaciones. La suma de todos
los $I_i$, convenientemente escalada, reconstruye el índice global.

Cada radio cae en uno de **cuatro cuadrantes**, que son los del diagrama de Moran:

| Cuadrante | Qué significa | Lectura |
|---|---|---|
| **HH** (alto-alto) | Valor alto rodeado de valores altos | Conglomerado de NBI alto |
| **LL** (bajo-bajo) | Valor bajo rodeado de valores bajos | Conglomerado de NBI bajo |
| **HL** (alto-bajo) | Valor alto rodeado de bajos | Caso atípico: una isla de NBI alto |
| **LH** (bajo-alto) | Valor bajo rodeado de altos | Caso atípico en sentido inverso |

Los dos primeros son **conglomerados**; los dos últimos, **valores atípicos espaciales**, y
suelen ser los más interesantes para una investigación social, porque señalan discontinuidades
bruscas del territorio.

> ⚠️ **Una advertencia estadística que corresponde declarar.** Acá se realizan 3.552 pruebas de
> hipótesis, una por radio. Al 5 % de significación, por puro azar se esperarían unos 178 falsos
> positivos. Los mapas LISA se leen como una herramienta **exploratoria**, para localizar zonas
> que merecen mirarse, y no como 3.552 afirmaciones independientes.

### ▶️ El cálculo

`seed` fija la semilla del generador aleatorio, de modo que las permutaciones den el mismo
resultado cada vez que se corra la notebook.

In [ ]:
lisa = esda.Moran_Local(y, w_queen, permutations=999, seed=0)

radios_m["lisa_p"] = lisa.p_sim
radios_m["lisa_q"] = lisa.q            # 1=HH, 2=LH, 3=LL, 4=HL

significativo = radios_m["lisa_p"] < 0.05
print(f"Radios con asociación local significativa: {significativo.sum()} de {len(radios_m)}")

### ▶️ Los cuatro tipos

A los radios que no alcanzan significación no se les asigna ninguna categoría: no es que no
tengan patrón, es que no hay evidencia suficiente para afirmarlo.

In [ ]:
ETIQUETAS = {1: "HH — alto rodeado de alto", 2: "LH — bajo rodeado de alto",
             3: "LL — bajo rodeado de bajo", 4: "HL — alto rodeado de bajo"}

radios_m["conglomerado"] = "No significativo"
for codigo, etiqueta in ETIQUETAS.items():
    radios_m.loc[significativo & (radios_m["lisa_q"] == codigo), "conglomerado"] = etiqueta

radios_m["conglomerado"].value_counts()

### 👀 El mapa de conglomerados

Los colores son los convencionales de este tipo de mapa: rojo para los conglomerados de valores
altos, azul para los de valores bajos, y tonos intermedios para los atípicos.

In [ ]:
COLORES = {
    "HH — alto rodeado de alto": "#c0392b",
    "LL — bajo rodeado de bajo": "#2c7bb6",
    "HL — alto rodeado de bajo": "#f4a582",
    "LH — bajo rodeado de alto": "#92c5de",
    "No significativo": "#ededed",
}

fig, eje = plt.subplots(figsize=(9, 10))
for etiqueta, color in COLORES.items():
    subconjunto = radios_m[radios_m["conglomerado"] == etiqueta]
    if len(subconjunto):
        subconjunto.plot(ax=eje, color=color, edgecolor="white",
                         linewidth=0.05, label=f"{etiqueta} ({len(subconjunto)})")

eje.legend(loc="upper left", fontsize=9, frameon=True)
eje.set_title("Conglomerados LISA del NBI — CABA 2022")
eje.set_axis_off()
plt.show()

### 🔍 Interpretación

De los 3.552 radios, **1.385 presentan asociación local significativa**: 402 conglomerados de
NBI alto, 856 de NBI bajo, y 127 valores atípicos.

El mapa muestra la estructura que el índice global resumía en un número. Los conglomerados **LL**
—más numerosos— forman un corredor continuo en el norte y el centro-norte de la ciudad. Los
**HH** se concentran en el sur y aparecen además como manchas compactas y bien delimitadas, que
corresponden a los barrios populares.

Lo más informativo para una investigación social son los **127 atípicos**. Un radio **HL**, con
NBI alto rodeado de vecinos bajos, marca una discontinuidad abrupta: un asentamiento dentro de
una zona acomodada. Los **LH** señalan lo inverso. En los dos casos el interés no está en el
valor sino en el contraste con el entorno, que es información que ni el mapa de coropletas de la
Clase 5 ni el índice global podían dar.

**El conglomerado es, además, una variable territorial más.** La columna `conglomerado` puede
incorporarse a la tabla de análisis como una categórica, y usarse en el trabajo final.

---

## 9. Join-Count: autocorrelación de una variable categórica

**Conceptos clave.** El I de Moran necesita una variable numérica: calcula medias y desvíos. Pero
buena parte de las variables de las ciencias sociales son **categóricas** —tiene o no tiene
cloacas, votó a un partido o a otro, es urbano o rural— y para esas hace falta otro estadístico.

El **Join-Count** es el más simple que existe, y su lógica se entiende sin fórmula. Se pinta cada
unidad de negro (**B**, la categoría 1) o de blanco (**W**, la categoría 0) y se cuentan las
**uniones**: cada par de unidades vecinas. Hay tres tipos posibles:

- **BB**: dos vecinas negras,
- **WW**: dos vecinas blancas,
- **BW**: una de cada color.

Si las categorías se repartieran al azar, la proporción de cada tipo sería predecible a partir de
cuántas unidades hay de cada color. **Un exceso de uniones BB y WW indica conglomerados**; un
exceso de BW indica un patrón alternado. La significación se evalúa igual que antes, por
permutaciones.

### ▶️ La variable binaria

Dividimos los radios por la mediana de la ciudad: la mitad con NBI más alto y la mitad con NBI
más bajo. El corte es una decisión, y con otro umbral el resultado cambiaría.

In [ ]:
mediana = radios_m["perc_nbi"].median()
radios_m["nbi_alto"] = (radios_m["perc_nbi"] > mediana).astype(int)

print(f"Mediana de la ciudad: {mediana:.2f} % de hogares con NBI")
print(radios_m["nbi_alto"].value_counts().rename({0: "por debajo (W)", 1: "por encima (B)"}).to_string())

In [ ]:
fig, eje = plt.subplots(figsize=(8, 9))

radios_m.plot(column="nbi_alto", ax=eje, categorical=True, cmap="binary",
              edgecolor="white", linewidth=0.05, legend=True,
              legend_kwds=dict(labels=["Por debajo de la mediana", "Por encima"], loc="upper left"))

eje.set_title("La variable convertida en dos categorías")
eje.set_axis_off()
plt.show()

### ▶️ El conteo de uniones

In [ ]:
jc = esda.Join_Counts(radios_m["nbi_alto"].values, w_queen, permutations=999)

print(f"Uniones entre radios vecinos: {round(jc.bb + jc.ww + jc.bw)}")
print(f"  de las cuales WW (bajo con bajo): {round(jc.ww)}")

pd.DataFrame({
    "observado": [round(jc.bb), round(jc.bw)],
    "esperado_al_azar": [round(jc.mean_bb), round(jc.mean_bw)],
}, index=["BB — alto con alto", "BW — alto con bajo"])

In [ ]:
print(f"Pseudo p-valor de las uniones BB: {jc.p_sim_bb:.3f}")

### 🔍 Interpretación

Se observan **4.152 uniones entre radios de NBI alto**, cuando al azar se esperarían unas 3.001.
Y solo **3.811 uniones entre categorías distintas**, contra las 6.006 esperadas: hay muchas menos
fronteras entre un radio alto y uno bajo de las que habría si las categorías se hubieran repartido
por sorteo.

Las dos cifras dicen lo mismo desde ángulos opuestos: **los radios de NBI alto tienden a estar
pegados entre sí**, y la ciudad está segmentada en zonas homogéneas con pocas transiciones.

Conviene retener que el resultado depende del corte elegido. Con la mediana, cada categoría
agrupa la mitad de los radios; con un umbral más exigente —el 10 % superior, por ejemplo— el
conteo de uniones BB sería mucho menor y mediría otra cosa. El umbral se elige por una razón
sustantiva y se declara.

---

## 10. El semivariograma: a qué distancia deja de haber parecido

**Conceptos clave.** El I de Moran usa una definición binaria de vecindad: o se es vecino o no.
El **semivariograma** prescinde de esa definición y trabaja con la distancia como una variable
continua. Responde una pregunta más fina: *¿hasta qué distancia dos unidades se siguen
pareciendo?*

Se construye así: se toman todos los pares posibles de unidades, se calcula para cada par la
distancia que los separa y la mitad de la diferencia al cuadrado de sus valores —la
**semivarianza**, de ahí el nombre—, y se promedia esa semivarianza por intervalos de distancia.

El gráfico resultante suele tener tres rasgos con nombre propio:

| Rasgo | Qué es | Qué indica |
|---|---|---|
| **Nugget** (pepita) | El valor al que tiende la curva cuando la distancia tiende a cero | Variabilidad a muy corta distancia: microescala y error de medición |
| **Sill** (meseta) | El valor en que la curva se estabiliza | La varianza total de la variable |
| **Rango** | La distancia a la que se alcanza la meseta | **Hasta dónde llega la dependencia espacial** |

El rango es el resultado sustantivo. Más allá de esa distancia, dos unidades no se parecen más
entre sí que dos unidades cualesquiera: la posición deja de informar.

### ▶️ El cálculo

Con 3.552 radios habría más de seis millones de pares, así que se trabaja sobre una muestra
aleatoria con semilla declarada. Cada radio se reduce a un punto interior, como en la Clase 6.

In [ ]:
from scipy.spatial.distance import pdist

puntos_m = radios_m.copy()
puntos_m["geometry"] = radios_m.representative_point()

generador = np.random.default_rng(0)
muestra = generador.choice(len(puntos_m), size=1500, replace=False)

coordenadas = np.c_[puntos_m.geometry.x.values[muestra], puntos_m.geometry.y.values[muestra]]
valores = y[muestra]

distancias = pdist(coordenadas)                              # en metros
semivarianzas = pdist(valores[:, None], "sqeuclidean") / 2

print(f"Pares de radios: {len(distancias):,}".replace(",", "."))
print(f"Distancia máxima: {distancias.max() / 1000:.1f} km")

Los intervalos se limitan a **la mitad de la distancia máxima**. Más allá quedan muy pocos pares
—solo los de esquina a esquina de la ciudad— y el promedio se vuelve inestable.

In [ ]:
limite = distancias.max() / 2
bordes = np.linspace(0, limite, 16)
indice = np.digitize(distancias, bordes)

curva = []
for b in range(1, len(bordes)):
    seleccion = indice == b
    if seleccion.sum() > 30:
        curva.append({"distancia_km": (bordes[b - 1] + bordes[b]) / 2000,
                      "semivarianza": semivarianzas[seleccion].mean(),
                      "pares": int(seleccion.sum())})

curva = pd.DataFrame(curva)
curva.head()

In [ ]:
fig, eje = plt.subplots(figsize=(9, 5))

eje.plot(curva["distancia_km"], curva["semivarianza"], "o-", color="#4c72b0")
eje.axhline(curva["semivarianza"].max(), color="grey", linestyle=":", linewidth=1)
eje.text(0.2, curva["semivarianza"].max() * 0.97, "meseta (sill)", fontsize=9, color="grey")
eje.axvline(3, color="#c44e52", linestyle="--", linewidth=1.2)
eje.text(3.1, curva["semivarianza"].min(), "rango ≈ 3 km", fontsize=9, color="#c44e52")

eje.set_xlabel("Distancia entre radios (km)")
eje.set_ylabel("Semivarianza")
eje.set_title("Semivariograma del NBI — CABA 2022")
eje.set_ylim(0, None)
plt.tight_layout()
plt.show()

### 🔍 Interpretación

La curva arranca en torno a **40** a distancias muy cortas y sube hasta estabilizarse alrededor
de **70** a partir de unos **3 kilómetros**.

Esos tres números se leen así. El **nugget** de 40 dice que dos radios contiguos ya difieren
bastante: la variabilidad a microescala es alta, lo que era esperable en una ciudad donde un
barrio popular puede lindar con una zona acomodada. El **sill** de 70 es la varianza total de la
variable. Y el **rango de 3 km** es el hallazgo: **la dependencia espacial del NBI se agota a los
tres kilómetros**.

Es una escala sustantiva, no un artefacto. En una ciudad que mide 17,6 km de punta a punta, tres
kilómetros es aproximadamente el tamaño de una comuna. Dicho de otro modo: para predecir el NBI
de un radio sirve mirar su entorno inmediato, y no sirve saber qué pasa en el otro extremo de la
ciudad.

Ese número tiene consecuencias prácticas. Es la escala a la que conviene definir una zona de
estudio, agregar unidades o diseñar una intervención territorial.

---

## 11. Asociación espacial y causalidad

Los cuatro estadísticos de hoy midieron lo mismo desde ángulos distintos, y coincidieron: el NBI
de la Ciudad de Buenos Aires **está espacialmente estructurado**. Eso está establecido y se puede
afirmar.

Lo que ninguno de los cuatro dice es **por qué**.

Un I de Moran de 0,543 es compatible con explicaciones que son incompatibles entre sí:

- **Herencia histórica.** La traza de la ciudad y la localización de la industria en el sur
  fijaron hace un siglo un patrón que se reproduce.
- **Segregación residencial.** El mercado de suelo ordena a la población por capacidad de pago, y
  los precios varían de manera continua en el espacio.
- **Difusión.** La condición de un hogar afecta a la de sus vecinos por mecanismos de contagio:
  redes de empleo, acceso a información, deterioro o mejora del entorno.
- **Confusión con una tercera variable.** El NBI se agrupa porque se agrupa la infraestructura de
  servicios, y es ésta la que explica ambos.

Las cuatro producen exactamente el mismo mapa LISA. **Distinguirlas no es un problema de
estadística espacial: es un problema de diseño de investigación**, y requiere datos que estos no
son —una serie temporal, una intervención, un instrumento, un límite administrativo arbitrario
que permita comparar—.

La afirmación que estos resultados sostienen es precisa y acotada: *el NBI de CABA presenta
autocorrelación espacial positiva significativa, con conglomerados de valores altos en el sur y
de valores bajos en el norte, y una dependencia espacial que se agota a los 3 km.* Todo lo que
vaya más allá de eso necesita otro diseño.

---

## 12. Cierre del seminario

### El recorrido de hoy

| Estadístico | Qué mide | Qué necesita | Resultado |
|---|---|---|---|
| **Matriz de pesos** | Quién es vecino de quién | Una decisión del analista | Queen, 6,8 vecinos promedio, sin islas |
| **I de Moran global** | Cuánta estructura hay en total | Variable numérica | 0,543 (p = 0,001) |
| **LISA** | Dónde están los conglomerados | Variable numérica | 1.385 radios significativos |
| **Join-Count** | Estructura de una variable categórica | Variable binaria | 4.152 uniones BB contra 3.001 esperadas |
| **Semivariograma** | Hasta qué distancia llega la dependencia | Distancias continuas | Rango ≈ 3 km |

### El recorrido del seminario

Las ocho clases fueron un solo trayecto.

Empezamos preguntando **por qué importa el dónde** y qué es un dato espacial. Vimos que ese dato
tiene particularidades que lo distinguen de cualquier otro —la unidad de análisis modificable, el
efecto de borde, la falacia ecológica—. Aprendimos a **conseguirlo** de fuentes abiertas, a
ponerlo en el **sistema de coordenadas** que corresponde según lo que se quiera medir, y a
**representarlo** sin que el mapa diga más de lo que el dato sostiene.

Después pasamos de consumir datos a **producirlos**: las cuatro familias de operaciones
espaciales, la accesibilidad por red, el cambio de unidad de análisis. Y hoy cerramos
**interrogando** la variable construida.

### La idea para llevarse

Todo el seminario estuvo atravesado por la misma advertencia, en formas distintas.

El mapa de la Clase 5 escondía decisiones: qué medida, qué clasificación, qué símbolo. La
variable de la Clase 6 también: qué radio, qué predicado, qué normalización. El cambio de unidad
de la Clase 7 introducía un supuesto de uniformidad que nadie ve en la columna final. Y el índice
de hoy depende de una matriz de vecindad que el analista eligió.

**Ninguna de esas decisiones queda escrita en el resultado.** Lo que distingue un trabajo
defendible de uno que no lo es no es la sofisticación de la técnica: es que esas decisiones estén
declaradas, y que las conclusiones no vayan más lejos de lo que los datos permiten.

### El trabajo final

La consigna completa está en
[`TRABAJO_FINAL.md`](https://github.com/renzoepolo/sig-ciencias-sociales/blob/main/TRABAJO_FINAL.md)
y en el aula virtual. Los conglomerados LISA que calculamos hoy son una de las variables
territoriales admitidas, y la sección de limitaciones —diez de los cien puntos— es donde entra
todo lo de la sección 11.

### Glosario

| Término | Definición |
|---|---|
| **Autocorrelación espacial** | Correlación de una variable consigo misma a través del espacio: el grado en que el valor de una unidad se parece al de sus vecinas. |
| **Matriz de pesos espaciales** ($W$) | Tabla que define quién es vecino de quién y con qué peso. La elige el analista. |
| **Contigüidad Queen / Rook** | Criterio de vecindad por frontera compartida: Queen admite el contacto en un vértice, Rook exige un segmento. |
| **Estandarización por filas** | Dividir cada peso por la cantidad de vecinos, de modo que el rezago espacial sea el promedio de los vecinos. |
| **Rezago espacial** | Promedio ponderado del valor de los vecinos de una unidad. |
| **I de Moran** | Índice global de autocorrelación espacial. Equivale a la pendiente del diagrama de Moran. |
| **Prueba por permutaciones** | Contraste que construye la distribución nula repartiendo los valores al azar sobre la misma geometría. |
| **LISA** | Descomposición local del índice global: un valor y un p-valor por unidad. |
| **Conglomerado / atípico espacial** | HH y LL son conglomerados; HL y LH, valores atípicos respecto de su entorno. |
| **Join-Count** | Estadístico de autocorrelación para variables categóricas: cuenta uniones entre vecinas del mismo y de distinto color. |
| **Semivariograma** | Curva de la semivarianza en función de la distancia. |
| **Nugget, sill, rango** | Variabilidad a distancia cero, varianza total, y distancia a la que se agota la dependencia espacial. |

### Tres preguntas para autoevaluarse

1. El I de Moran de una variable da 0,02 con un p-valor de 0,4. ¿Qué se puede concluir, y qué
   no?
2. Un mapa LISA muestra 200 radios significativos al 5 % sobre un total de 4.000. ¿Alcanza ese
   número para afirmar que hay conglomerados? ¿Qué habría que comparar?
3. Dos investigadores analizan la misma variable sobre las mismas unidades y obtienen índices de
   Moran distintos. ¿Cuál es la primera hipótesis sobre la diferencia?

---

## 13. Referencias

- Anselin, L. (1995). "Local Indicators of Spatial Association — LISA". *Geographical Analysis*,
  27(2), 93–115.
- Moran, P. A. P. (1950). "Notes on Continuous Stochastic Phenomena". *Biometrika*, 37(1/2),
  17–23.
- Tobler, W. (1970). "A Computer Movie Simulating Urban Growth in the Detroit Region".
  *Economic Geography*, 46, 234–240.
- Rey, S., Arribas-Bel, D. y Wolf, L. J. (2023). *Geographic Data Science with Python*, caps. 6
  "Spatial Autocorrelation" y 7 "Local Spatial Autocorrelation". CRC Press.
- de Smith, M. J., Goodchild, M. F. y Longley, P. A. (2018). *Geospatial Analysis*, cap. 5
  "Exploratory Spatial Data Analysis".
- Dirección General de Estadística y Censos del Gobierno de la Ciudad de Buenos Aires (2023).
  *Censo Nacional de Población, Hogares y Viviendas 2022. Resultados por radio censal*.